# 🚗 Car Insurance Fraud Detection — Improved Model

---

## 📖 What's new in this version?

This notebook builds on the original project and adds 4 professional improvements:

| Improvement | What it does |
|---|---|
| **SMOTE** | Fixes the imbalanced dataset problem (more fraud examples to learn from) |
| **Cross-Validation** | More reliable evaluation than a single train/test split |
| **XGBoost & LightGBM** | Industry-standard models, usually more accurate than Random Forest |
| **GridSearchCV** | Automatically finds the best settings for each model |

---

## 🗺️ Roadmap

```
1.  Install & Import
2.  Load Data
3.  Preprocess (same as before)
4.  SMOTE — fix class imbalance
5.  Cross-Validation — robust evaluation
6.  Train XGBoost & LightGBM
7.  Hyperparameter Tuning with GridSearchCV
8.  Final Evaluation & Comparison
9.  Save Best Model with Pickle
```

---
## Step 1: Install & Import Libraries

We need two new libraries that don't come with Colab by default:
- `imbalanced-learn` → gives us SMOTE
- `lightgbm` → gives us the LightGBM model

XGBoost is already installed in Colab.

In [ ]:
# Install new libraries (only needed in Colab)
!pip install imbalanced-learn lightgbm --quiet
print('✅ Installation done!')

In [ ]:
# ─── Data Handling ────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

# ─── Visualization ────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')

# ─── Preprocessing ────────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler

# ─── SMOTE (fix imbalanced data) ──────────────────────────────────────────────
from imblearn.over_sampling import SMOTE

# ─── Models ───────────────────────────────────────────────────────────────────
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# ─── Evaluation ───────────────────────────────────────────────────────────────
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, roc_auc_score, roc_curve
)

print('✅ All libraries imported!')

---
## Step 2: Load & Preprocess Data

Same preprocessing as the original notebook — drop useless columns, encode categories, split, and scale.

In [ ]:
data = pd.read_csv('/content/car_insurance_fraud_dataset.csv')
print(f'Dataset shape: {data.shape}')
data.head(3)

In [ ]:
# ── Drop non-useful columns ───────────────────────────────────────────────────
df = data.drop(columns=['policy_id', 'incident_date', 'incident_city'])

# ── Encode target ─────────────────────────────────────────────────────────────
if df['fraud_reported'].dtype == object:
    df['fraud_reported'] = df['fraud_reported'].map({'Y': 1, 'N': 0})

# ── Encode categorical columns ────────────────────────────────────────────────
categorical_cols = df.select_dtypes(include='object').columns.tolist()
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

# ── Split features and target ─────────────────────────────────────────────────
X = df.drop(columns=['fraud_reported'])
y = df['fraud_reported']

# ── Train/Test split ──────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ── Scale ─────────────────────────────────────────────────────────────────────
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f'Train size: {X_train.shape[0]}  |  Test size: {X_test.shape[0]}')
print(f'Fraud % in train: {y_train.mean()*100:.1f}%')
print('✅ Preprocessing done!')

---
## Step 3: SMOTE — Fix the Imbalanced Dataset Problem

### What is the imbalance problem?

Look at our dataset — roughly **~80% legitimate** and **~20% fraud**.

This is a problem because the model can cheat:
> It can just predict "LEGITIMATE" for every single claim and still get 80% accuracy!
> But it would catch ZERO fraud cases — completely useless.

This is called **class imbalance** — one class has far more examples than the other.

### What is SMOTE?

**SMOTE** = Synthetic Minority Over-sampling Technique

Instead of just duplicating fraud cases (which doesn't add new information), SMOTE **creates brand new synthetic fraud examples** by interpolating between existing ones.

```
Before SMOTE:  80% Legitimate  |  20% Fraud
After SMOTE:   50% Legitimate  |  50% Fraud   ← balanced!
```

### ⚠️ Critical Rule:
> SMOTE must ONLY be applied to the **training set**, never the test set.
> The test set must stay real — no synthetic data.
> If you apply SMOTE to test data, your evaluation results will be fake.

In [ ]:
# ── Check class distribution BEFORE SMOTE ─────────────────────────────────────
print('Before SMOTE:')
print(f'  Legitimate (0): {sum(y_train == 0):,}  ({sum(y_train==0)/len(y_train)*100:.1f}%)')
print(f'  Fraud      (1): {sum(y_train == 1):,}  ({sum(y_train==1)/len(y_train)*100:.1f}%)')

# ── Apply SMOTE ───────────────────────────────────────────────────────────────
# random_state=42 → reproducible results
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

# ── Check class distribution AFTER SMOTE ──────────────────────────────────────
print('\nAfter SMOTE:')
print(f'  Legitimate (0): {sum(y_train_smote == 0):,}  ({sum(y_train_smote==0)/len(y_train_smote)*100:.1f}%)')
print(f'  Fraud      (1): {sum(y_train_smote == 1):,}  ({sum(y_train_smote==1)/len(y_train_smote)*100:.1f}%)')
print(f'\nTotal training samples: {len(y_train)} → {len(y_train_smote)}')

# ── Visualize the change ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Class Distribution: Before vs After SMOTE', fontsize=13, fontweight='bold')

pd.Series(y_train).value_counts().plot(
    kind='bar', ax=axes[0], color=['steelblue','tomato'], edgecolor='black')
axes[0].set_title('Before SMOTE', fontweight='bold')
axes[0].set_xticklabels(['Legitimate','Fraud'], rotation=0)
axes[0].set_ylabel('Count')

pd.Series(y_train_smote).value_counts().plot(
    kind='bar', ax=axes[1], color=['steelblue','tomato'], edgecolor='black')
axes[1].set_title('After SMOTE (Balanced)', fontweight='bold')
axes[1].set_xticklabels(['Legitimate','Fraud'], rotation=0)

plt.tight_layout()
plt.show()

---
## Step 4: Cross-Validation — More Reliable Evaluation

### What's wrong with a single train/test split?

When we split data once (80% train, 20% test), we get lucky or unlucky depending on WHICH 20% ended up in the test set. The result can vary significantly.

### What is Cross-Validation?

Instead of splitting once, we split the data **5 times** (called folds) and train+test 5 separate times:

```
Fold 1: [TEST ] [train] [train] [train] [train]  → score 1
Fold 2: [train] [TEST ] [train] [train] [train]  → score 2
Fold 3: [train] [train] [TEST ] [train] [train]  → score 3
Fold 4: [train] [train] [train] [TEST ] [train]  → score 4
Fold 5: [train] [train] [train] [train] [TEST ]  → score 5
                                                   ─────────
                                         Final score = average
```

This gives a much more **honest and stable** estimate of model performance.

### What is StratifiedKFold?

Regular KFold splits randomly. **StratifiedKFold** makes sure each fold has the **same fraud ratio** as the full dataset. This is essential for imbalanced problems.

In [ ]:
# ── Define 5-fold stratified cross-validation ─────────────────────────────────
# n_splits=5 → 5 folds
# shuffle=True → shuffle data before splitting
# random_state=42 → reproducible
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ── Models to evaluate ────────────────────────────────────────────────────────
cv_models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost':       XGBClassifier(n_estimators=100, random_state=42,
                                   eval_metric='logloss', verbosity=0),
    'LightGBM':      LGBMClassifier(n_estimators=100, random_state=42, verbose=-1)
}

print('Running 5-Fold Cross-Validation on SMOTE-balanced data...')
print('(This takes a minute — each model trains 5 times)\n')

cv_results = {}

for name, model in cv_models.items():
    # cross_val_score runs the full train/test cycle n_splits times
    # scoring='roc_auc' → we measure ROC-AUC each time
    scores = cross_val_score(
        model, X_train_smote, y_train_smote,
        cv=cv, scoring='roc_auc', n_jobs=-1  # n_jobs=-1 = use all CPU cores
    )
    cv_results[name] = scores
    print(f'{name:<20} | Mean AUC: {scores.mean():.4f} | Std: {scores.std():.4f} | Scores: {np.round(scores, 4)}')

print('\n✅ Cross-validation done!')
print('\n💡 Lower std = more stable/consistent model')

In [ ]:
# ── Visualize cross-validation results ────────────────────────────────────────
# Box plot shows the spread of scores across all 5 folds
# A tight box = consistent model, a wide box = unstable model

plt.figure(figsize=(10, 5))
data_to_plot = [cv_results[name] for name in cv_results]
bp = plt.boxplot(data_to_plot, labels=list(cv_results.keys()),
                 patch_artist=True, widths=0.5)

colors = ['steelblue', 'darkorange', 'green']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

plt.ylabel('ROC-AUC Score')
plt.title('5-Fold Cross-Validation Results', fontsize=13, fontweight='bold')
plt.ylim(0.5, 1.05)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Step 5: XGBoost & LightGBM — Industry Models

### What is XGBoost?

**XGBoost** = Extreme Gradient Boosting.

Unlike Random Forest which builds many trees **independently** and averages them, XGBoost builds trees **sequentially** — each new tree focuses on fixing the mistakes of the previous one. This is called **boosting**.

```
Random Forest:  Tree1 + Tree2 + Tree3 + ...  → average all predictions
XGBoost:        Tree1 → fix errors → Tree2 → fix errors → Tree3 → ...  → final prediction
```

XGBoost won dozens of Kaggle competitions and is used heavily in industry for tabular data.

### What is LightGBM?

**LightGBM** = Light Gradient Boosting Machine. Created by Microsoft.

It works like XGBoost but is **much faster** on large datasets because it grows trees differently (leaf-wise instead of level-wise). For 30,000 rows it might not feel different, but on millions of rows LightGBM can be 10x faster.

In practice: **LightGBM is the first model most data scientists try** on tabular data.

In [ ]:
# ── Train all 3 models on SMOTE-balanced training data ────────────────────────
final_models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost':       XGBClassifier(n_estimators=100, random_state=42,
                                   eval_metric='logloss', verbosity=0),
    'LightGBM':      LGBMClassifier(n_estimators=100, random_state=42, verbose=-1)
}

results = {}

for name, model in final_models.items():
    # Train on SMOTE data
    model.fit(X_train_smote, y_train_smote)

    # Evaluate on REAL test data (no SMOTE applied here!)
    y_pred  = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)

    results[name] = {'accuracy': acc, 'roc_auc': auc,
                     'y_pred': y_pred, 'y_proba': y_proba,
                     'model': model}

    print(f'{name:<20} Accuracy: {acc:.4f}  |  ROC-AUC: {auc:.4f}')

print('\n✅ All models trained on SMOTE-balanced data!')

---
## Step 6: Hyperparameter Tuning with GridSearchCV

### What is a hyperparameter?

A **parameter** is what the model learns automatically during training (like the weights in each tree).

A **hyperparameter** is a setting you choose BEFORE training — like:
- How many trees to build? (`n_estimators`)
- How deep can each tree go? (`max_depth`)
- How fast should the model learn? (`learning_rate`)

The default values are good, but the BEST values depend on your specific dataset.

### What is GridSearchCV?

**GridSearchCV** tries every possible combination of hyperparameters you give it and uses cross-validation to find which combination gives the best score.

```
You give it:    n_estimators = [100, 200]
                max_depth    = [3, 5, 7]

It tries:       100 trees + depth 3  → CV score
                100 trees + depth 5  → CV score
                100 trees + depth 7  → CV score
                200 trees + depth 3  → CV score
                200 trees + depth 5  → CV score
                200 trees + depth 7  → CV score
                                       ─────────
                             Best combination wins!
```

### ⚠️ Why we tune only LightGBM here:
GridSearchCV is slow because it trains the model many times.
We focus on LightGBM because it's the fastest model — tuning XGBoost or
Random Forest the same way would take much longer in Colab.

In [ ]:
# ── Define the hyperparameter grid ────────────────────────────────────────────
# These are the combinations GridSearchCV will try
param_grid = {
    'n_estimators':  [100, 200],       # how many trees
    'max_depth':     [3, 5, 7],        # max depth of each tree
    'learning_rate': [0.05, 0.1, 0.2], # how fast the model learns
    'num_leaves':    [31, 63]          # LightGBM-specific: max leaves per tree
}
# Total combinations: 2 × 3 × 3 × 2 = 36 combinations × 3 CV folds = 108 fits
print(f'GridSearch will try {2*3*3*2} combinations × 3 folds = {2*3*3*2*3} model fits')
print('This will take a few minutes...\n')

# ── Run GridSearchCV ──────────────────────────────────────────────────────────
lgbm_base = LGBMClassifier(random_state=42, verbose=-1)

grid_search = GridSearchCV(
    estimator  = lgbm_base,
    param_grid = param_grid,
    cv         = 3,              # 3-fold CV (faster than 5-fold for tuning)
    scoring    = 'roc_auc',      # optimize for ROC-AUC
    n_jobs     = -1,             # use all CPU cores
    verbose    = 1               # show progress
)

grid_search.fit(X_train_smote, y_train_smote)

# ── Results ───────────────────────────────────────────────────────────────────
print(f'\n✅ Best hyperparameters found:')
for param, value in grid_search.best_params_.items():
    print(f'   {param}: {value}')
print(f'\nBest CV ROC-AUC: {grid_search.best_score_:.4f}')

In [ ]:
# ── Evaluate the tuned model on the test set ──────────────────────────────────
best_lgbm = grid_search.best_estimator_

y_pred_tuned  = best_lgbm.predict(X_test_scaled)
y_proba_tuned = best_lgbm.predict_proba(X_test_scaled)[:, 1]

acc_tuned = accuracy_score(y_test, y_pred_tuned)
auc_tuned = roc_auc_score(y_test, y_proba_tuned)

# Add to results for final comparison
results['LightGBM (Tuned)'] = {
    'accuracy': acc_tuned,
    'roc_auc':  auc_tuned,
    'y_pred':   y_pred_tuned,
    'y_proba':  y_proba_tuned,
    'model':    best_lgbm
}

print(f'LightGBM (default)  →  AUC: {results["LightGBM"]["roc_auc"]:.4f}')
print(f'LightGBM (tuned)    →  AUC: {auc_tuned:.4f}')
improvement = (auc_tuned - results['LightGBM']['roc_auc']) * 100
print(f'Improvement: {improvement:+.2f}%')

---
## Step 7: Final Evaluation & Comparison

Now we compare ALL models side by side on the real test set.

In [ ]:
# ── Confusion Matrices ────────────────────────────────────────────────────────
n_models = len(results)
fig, axes = plt.subplots(1, n_models, figsize=(5 * n_models, 5))
fig.suptitle('Confusion Matrices — All Models', fontsize=14, fontweight='bold')

for ax, (name, res) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, res['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Legitimate', 'Fraud'],
                yticklabels=['Legitimate', 'Fraud'],
                linewidths=0.5)
    ax.set_title(name, fontweight='bold', fontsize=10)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.show()

In [ ]:
# ── Classification Reports ────────────────────────────────────────────────────
for name, res in results.items():
    print(f'{'='*55}\n {name}\n{'='*55}')
    print(classification_report(y_test, res['y_pred'],
                                target_names=['Legitimate', 'Fraud']))

In [ ]:
# ── ROC Curves ────────────────────────────────────────────────────────────────
plt.figure(figsize=(10, 6))
colors = ['steelblue', 'darkorange', 'green', 'crimson']

for (name, res), color in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, res['y_proba'])
    plt.plot(fpr, tpr, color=color, lw=2,
             label=f"{name}  (AUC = {res['roc_auc']:.4f})")

plt.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random (AUC = 0.50)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.title('ROC Curves — All Models', fontsize=13, fontweight='bold')
plt.legend(loc='lower right', fontsize=9)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── Final Summary Table ───────────────────────────────────────────────────────
summary = pd.DataFrame({
    'Model':    list(results.keys()),
    'Accuracy': [res['accuracy'] for res in results.values()],
    'ROC-AUC':  [res['roc_auc']  for res in results.values()],
    'CV Mean AUC': [
        cv_results.get(name, {0: None}).mean() if name in cv_results else '(tuned)'
        for name in results.keys()
    ]
}).sort_values('ROC-AUC', ascending=False).reset_index(drop=True)

print('📊 Final Model Comparison:')
print(summary.to_string(index=False))

best_model_name = summary.iloc[0]['Model']
best_auc        = summary.iloc[0]['ROC-AUC']
print(f'\n🏆 Best model: {best_model_name}  (AUC = {best_auc:.4f})')

In [ ]:
# ── Feature Importance (LightGBM Tuned) ───────────────────────────────────────
# feature_importances_ shows how much each feature contributed to the model's decisions
best_model_obj = results[best_model_name]['model']

importances = pd.Series(
    best_model_obj.feature_importances_,
    index=X.columns
).sort_values(ascending=False).head(15)

plt.figure(figsize=(10, 6))
plt.barh(importances.index[::-1], importances.values[::-1],
         color='steelblue', edgecolor='black')
plt.xlabel('Importance Score')
plt.title(f'Top 15 Features — {best_model_name}', fontweight='bold')
plt.tight_layout()
plt.show()

---
## Step 8: Save the Best Model with Pickle

Same as before — we save the model, scaler, label encoders, and feature columns.
These 4 files are what the FastAPI server needs to make predictions.

In [ ]:
# ── ALWAYS verify types before saving ────────────────────────────────────────
# This prevents the 'StandardScaler has no attribute items' bug from before!
print('Verifying objects before saving...')
print(f'  label_encoders → {type(label_encoders)}')
print(f'  scaler         → {type(scaler)}')
print(f'  best model     → {type(results[best_model_name]["model"])}')

assert isinstance(label_encoders, dict), '❌ label_encoders is not a dict!'
assert hasattr(scaler, 'transform'),     '❌ scaler is not a StandardScaler!'
print('\n✅ All objects verified — safe to save!')

In [ ]:
os.makedirs('/content/model_artifacts', exist_ok=True)

best_model_to_save = results[best_model_name]['model']

# Save in the CORRECT order — variable name matches file name!
with open('/content/model_artifacts/model.pkl', 'wb') as f:
    pickle.dump(best_model_to_save, f)

with open('/content/model_artifacts/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

with open('/content/model_artifacts/label_encoders.pkl', 'wb') as f:
    pickle.dump(label_encoders, f)

with open('/content/model_artifacts/feature_columns.pkl', 'wb') as f:
    pickle.dump(list(X.columns), f)

print(f'✅ Saved best model: {best_model_name}')
print('   → model.pkl')
print('   → scaler.pkl')
print('   → label_encoders.pkl')
print('   → feature_columns.pkl')

In [ ]:
# ── Verify the saved model works correctly ────────────────────────────────────
with open('/content/model_artifacts/model.pkl', 'rb') as f:
    loaded_model = pickle.load(f)
with open('/content/model_artifacts/scaler.pkl', 'rb') as f:
    loaded_scaler = pickle.load(f)

y_pred_loaded = loaded_model.predict(loaded_scaler.transform(X_test))
acc_loaded    = accuracy_score(y_test, y_pred_loaded)

print(f'✅ Loaded model accuracy: {acc_loaded:.4f}')
print(f'   Original accuracy:     {results[best_model_name]["accuracy"]:.4f}')
print('   → Match! Model saved and loaded correctly.')

In [ ]:
# ── Download all pickle files ─────────────────────────────────────────────────
from google.colab import files
for fname in ['model.pkl', 'scaler.pkl', 'label_encoders.pkl', 'feature_columns.pkl']:
    files.download(f'/content/model_artifacts/{fname}')
    print(f'⬇️  Downloading {fname}...')

---
## 📝 What You Learned in This Notebook

| Concept | Plain English |
|---|---|
| **Class Imbalance** | When one class has far more examples than the other (80% legit vs 20% fraud) |
| **SMOTE** | Creates synthetic minority examples to balance the dataset — only applied to training data |
| **Cross-Validation** | Trains and tests 5 times on different splits, gives more honest performance estimate |
| **StratifiedKFold** | Like KFold but keeps the fraud ratio the same in each fold |
| **XGBoost** | Builds trees sequentially, each fixing the previous one's mistakes |
| **LightGBM** | Same idea as XGBoost but much faster, great first choice for tabular data |
| **Hyperparameter** | A setting you choose before training (depth, learning rate, trees) |
| **GridSearchCV** | Automatically tries every combination of hyperparameters to find the best ones |
| **Overfitting** | Model memorizes training data, fails on new data — tuning helps prevent this |

### What to learn next
- **RandomizedSearchCV** — like GridSearch but samples randomly, much faster on large grids
- **Optuna** — smarter hyperparameter search used in production
- **SHAP values** — explains WHY the model made each specific prediction
- **Calibration** — makes the fraud probability score more accurate and trustworthy